# 04 · Clasificación multiclase con un dataset real (Wine)## ObjetivoAplicar todo lo aprendido en los notebooks anteriores a un **dataset real** con más de 2 clases, usando Scikit-learn para los datos y TensorFlow/Keras para el modelo — el mismo patrón que vimos en el tema "Scikit-learn vs TensorFlow": cada biblioteca hace lo que mejor sabe hacer.Usaremos el dataset **Wine** (vinos) de scikit-learn, un dataset clásico similar en espíritu al de Iris, pero con distintos datos: características químicas de vinos, clasificados en 3 tipos de cultivo.## Teoría: el flujo completo de un proyecto de clasificación multiclase1. Cargar el dataset.2. Separar en datos de entrenamiento y de prueba (`train_test_split`), para poder evaluar el modelo con datos que nunca vio.3. Definir la arquitectura: la capa de entrada debe tener tantas neuronas como **características** tenga el dataset, y la capa de salida tantas neuronas como **clases** posibles, con activación `softmax`.4. Compilar, entrenar, y evaluar con `argmax` (la clase con mayor probabilidad).

## Paso 1: Importar libreríasAquí se ve claramente la combinación Scikit-learn + TensorFlow:- `sklearn.datasets.load_wine`: para obtener el dataset real.- `sklearn.model_selection.train_test_split`: para dividir los datos.- `tensorflow` / `keras`: para construir y entrenar la red neuronal.

In [ ]:
import tensorflow as tfimport numpy as npfrom sklearn.datasets import load_winefrom sklearn.model_selection import train_test_split

## Paso 2: Cargar y explorar el dataset- `X = wine.data`: las características (2D — una fila por vino, una columna por característica química).- `Y = wine.target`: las etiquetas (1D — el tipo de vino, como número entero: 0, 1 o 2).Antes de construir el modelo, siempre conviene mirar cuántas características y cuántas clases hay — esos números definen el tamaño de la capa de entrada y de salida.

In [ ]:
wine = load_wine()X = wine.dataY = wine.targetprint("Forma de X (filas, columnas):", X.shape)print("Cantidad de características (para la capa de entrada):", X.shape[1])print("Clases posibles:", wine.target_names)print("Cantidad de clases (para la capa de salida):", len(wine.target_names))

## Paso 3: Separar en entrenamiento y prueba`train_test_split` reparte los datos en dos grupos:- **80% entrenamiento**: con lo que el modelo aprende.- **20% prueba**: datos que el modelo NUNCA ve durante el entrenamiento, para evaluar si realmente aprendió o solo "memorizó".`random_state=42` fija la semilla aleatoria, para que el split sea siempre el mismo si volvemos a correr el notebook (reproducibilidad).

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(    X, Y, test_size=0.2, random_state=42)print("Datos de entrenamiento:", X_train.shape[0])print("Datos de prueba:", X_test.shape[0])

## Paso 4: Construir la arquitectura- Capa de entrada implícita: `input_shape=(13,)` porque el dataset Wine tiene 13 características químicas por vino.- Dos capas ocultas con ReLU (12 y 8 neuronas), para darle más capacidad de aprendizaje que en los notebooks anteriores.- Capa de salida: 3 neuronas (una por tipo de vino) con activación **softmax**, que convierte las salidas en una distribución de probabilidad que suma 1 entre las 3 clases (a diferencia de sigmoid, que da una sola probabilidad para 2 clases).

In [ ]:
modelo = tf.keras.models.Sequential([    tf.keras.layers.Dense(12, activation="relu", input_shape=(13,)),    tf.keras.layers.Dense(8, activation="relu"),    tf.keras.layers.Dense(3, activation="softmax")])modelo.summary()

## Paso 5: Compilar- **loss="sparse_categorical_crossentropy"**: se usa cuando las etiquetas son números enteros (0, 1, 2), como en nuestro caso. Si las etiquetas estuvieran en formato one-hot (ej. [1,0,0]), usaríamos `categorical_crossentropy` en su lugar.

In [ ]:
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

## Paso 6: EntrenarEntrenamos solo con los datos de entrenamiento (`X_train`, `Y_train`), dejando el conjunto de prueba completamente aparte.

In [ ]:
historial = modelo.fit(X_train, Y_train, epochs=100, verbose=0)print("Pérdida final:", historial.history["loss"][-1])print("Exactitud (accuracy) final:", historial.history["accuracy"][-1])

## Paso 7: Predecir una muestra individual`softmax` entrega una probabilidad por cada una de las 3 clases. `argmax` nos dice cuál de esas 3 probabilidades es la más alta — esa es la clase que el modelo elige.

In [ ]:
muestra = X_test[0].reshape(1, 13)etiqueta_real = Y_test[0]prediccion = modelo.predict(muestra)print("Probabilidades por clase:", prediccion)print("Clase predicha:", prediccion.argmax(1))print("Clase real:", etiqueta_real)

## Paso 8: Evaluar sobre todo el conjunto de pruebaComparamos, de un vistazo, todas las predicciones contra todas las etiquetas reales del conjunto de prueba (el 20% que el modelo nunca vio entrenando).

In [ ]:
predicciones = modelo.predict(X_test).argmax(1)print("Predicciones:", predicciones)print("Etiquetas reales:", Y_test)print("Aciertos:", (predicciones == Y_test).sum(), "de", len(Y_test))

## Conclusión del notebook- Vimos el ciclo completo de un proyecto de clasificación multiclase con un dataset real: cargar → dividir → construir → compilar → entrenar → evaluar.- Scikit-learn se encargó de los datos (dataset + split); TensorFlow/Keras se encargó del modelo (red neuronal).- La arquitectura de salida (`softmax` + `sparse_categorical_crossentropy`) es la misma "plantilla multiclase" que vimos en el notebook 03, aplicada ahora a un caso real.